# 🔎 API REST de contenido SEO — FastAPI + OpenAI SDK

**Reto proyecto — Desarrollo de Soluciones IA**

API REST con **FastAPI** que integra el **SDK de OpenAI** (cliente de Azure) para generar contenido SEO con **salidas estructuradas Pydantic**: keywords, artículos con estructura H1/H2/H3, metadatos (60/160 caracteres), FAQs con **JSON-LD FAQPage** y resúmenes por red social (Twitter/X, LinkedIn, Instagram, Facebook). Implementa la plantilla oficial del reto (mismos archivos, mismos nombres).

### Mapeo estructura del proyecto → celdas

```
seo-content-api/
├── app/
│   ├── config.py            → Celda «Configuración»
│   ├── models/
│   │   ├── keywords.py      → Celda «Modelos: keywords»
│   │   ├── articles.py      → Celda «Modelos: articles»
│   │   ├── metadata.py      → Celda «Modelos: metadata»
│   │   ├── faqs.py          → Celda «Modelos: faqs»
│   │   └── social.py        → Celda «Modelos: social»
│   ├── services/seo_service.py → Celda «Servicio SEO»
│   ├── routers/*.py         → Celda «Routers»
│   └── main.py              → Celda «App FastAPI»
├── .env                     → Credenciales (privadas)
├── requirements.txt         → Celda de dependencias
└── README.md                → Este encabezado
```

### Los 5 endpoints

| Endpoint | Request | Response |
|---|---|---|
| `POST /api/keywords/generate` | KeywordRequest | KeywordResponse |
| `POST /api/articles/generate` | ArticleRequest | ArticleResponse |
| `POST /api/metadata/generate` | MetadataRequest | MetadataResponse |
| `POST /api/faqs/extract` | FAQRequest | FAQResponse |
| `POST /api/social/summaries` | SocialRequest | SocialResponse |

> En `/api/social/summaries`, un campo de plataforma a `null` significa: o no se solicitó, o su generación falló (fallo parcial — el resto de plataformas se devuelve igualmente). El detalle del fallo queda en los logs del servidor.

### Configuración (`.env`)

```dotenv
AZURE_OPENAI_API_KEY=tu_clave_de_azure
AZURE_OPENAI_ENDPOINT=https://marcvancutseme7172-2656-resource.services.ai.azure.com/
AZURE_OPENAI_CLIENT=openai      # 'openai' (endpoint v1, gpt-5) o 'azure' (AzureOpenAI con api_version)
OPENAI_MODEL=gpt-5
OPENAI_API_VERSION=2024-10-21   # solo se usa con AZURE_OPENAI_CLIENT=azure
```

> **Nota del enunciado:** con **Azure for students** solo hay `gpt-4o` / `gpt-4o-mini`; en ese caso usa `AZURE_OPENAI_CLIENT=azure` y `OPENAI_MODEL=gpt-4o-mini`. Con **tu recurso propio** (como aquí), `openai` + `gpt-5` funciona y es lo configurado por defecto. Ambos caminos usan el cliente de Azure del enunciado (`AzureOpenAI`) o el compatible v1 del mismo recurso.

### Cómo ejecutar
Celdas de arriba abajo: dependencias → config → modelos → servicio → routers → app → **arrancar servidor en localhost** → demos de los 5 endpoints. La documentación **Swagger** queda en `http://127.0.0.1:8010/docs`.


## 1. Dependencias (`requirements.txt`)

```text
fastapi
uvicorn
openai
pydantic
python-dotenv
requests
```


In [17]:
# Instalación de dependencias (ejecutar una vez)
%pip install -q fastapi uvicorn openai pydantic python-dotenv requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración (`app/config.py`)

Variables de entorno y **cliente OpenAI**. Dos modos:
- `AZURE_OPENAI_CLIENT=openai`: cliente `OpenAI` sobre el endpoint compatible `/openai/v1/` del recurso de Azure (tu caso, con `gpt-5`).
- `AZURE_OPENAI_CLIENT=azure`: cliente **`AzureOpenAI`** clásico con `api_version` (Azure for students, `gpt-4o`/`gpt-4o-mini`).


In [18]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
_raw = os.getenv("AZURE_OPENAI_ENDPOINT",
                 "https://marcvancutseme7172-2656-resource.services.ai.azure.com/")
AZURE_OPENAI_ENDPOINT = _raw.rstrip("/")
for _suf in ("/openai/v1", "/openai"):
    if AZURE_OPENAI_ENDPOINT.endswith(_suf):
        AZURE_OPENAI_ENDPOINT = AZURE_OPENAI_ENDPOINT[: -len(_suf)]
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION", "2024-10-21")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5")
CLIENT_MODE = os.getenv("AZURE_OPENAI_CLIENT", "openai").lower()   # 'openai' | 'azure'

# Host/puerto del servidor de la API en el notebook
API_HOST = os.getenv("API_HOST", "127.0.0.1")
API_PORT = int(os.getenv("API_PORT", "8010"))
API_URL = f"http://{API_HOST}:{API_PORT}"

_cliente = None


def get_client() -> OpenAI:
    """Devuelve el cliente adecuado según la configuración (cacheado)."""
    global _cliente
    if _cliente is None:
        if CLIENT_MODE == "azure":
            # Cliente de Azure clásico (Azure for students: gpt-4o / gpt-4o-mini)
            _cliente = AzureOpenAI(
                api_key=AZURE_OPENAI_API_KEY,
                azure_endpoint=AZURE_OPENAI_ENDPOINT,
                api_version=OPENAI_API_VERSION,
            )
        else:
            # Endpoint compatible OpenAI v1 del MISMO recurso de Azure (gpt-5)
            _cliente = OpenAI(
                api_key=AZURE_OPENAI_API_KEY,
                base_url=AZURE_OPENAI_ENDPOINT + "/openai/v1/",
            )
    return _cliente


print("✅ Configuración cargada.")
print(f"   Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"   Cliente:  {CLIENT_MODE}  | Modelo: {OPENAI_MODEL}  | Clave: {'✅' if AZURE_OPENAI_API_KEY else '❌'}")
print(f"   API:      {API_URL}  (docs en {API_URL}/docs)")


✅ Configuración cargada.
   Endpoint: https://marcvancutseme7172-2656-resource.services.ai.azure.com
   Cliente:  openai  | Modelo: gpt-5  | Clave: ✅
   API:      http://127.0.0.1:8010  (docs en http://127.0.0.1:8010/docs)


## 3. Modelos: keywords (`app/models/keywords.py`)

Entrada con valores por defecto; salida con listas de keywords y la clasificación de intención como **lista de submodelos** (`KeywordIntent`), siguiendo el consejo de la plantilla: nada de diccionarios libres, y `extra="forbid"` en los modelos de respuesta (requisito del modo estricto de structured outputs).


In [19]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field


class KeywordRequest(BaseModel):
    """Entrada del endpoint de keywords."""
    topic: str = Field(description="Tema principal sobre el que generar keywords")
    industry: str = Field(default="general", description="Sector o industria del contenido")
    language: str = Field(default="es", description="Idioma de las keywords (código ISO)")


class KeywordIntent(BaseModel):
    """Clasificación de intención de una keyword concreta."""
    model_config = ConfigDict(extra="forbid")
    keyword: str = Field(description="La keyword clasificada")
    intent: Literal["informacional", "transaccional"] = Field(
        description="Intención de búsqueda de la keyword"
    )


class KeywordResponse(BaseModel):
    """Salida del endpoint de keywords."""
    model_config = ConfigDict(extra="forbid")
    seed_keywords: list[str] = Field(description="Keywords semilla (términos base, 1-2 palabras)")
    long_tail_keywords: list[str] = Field(description="Variantes long-tail (3+ palabras, específicas)")
    questions: list[str] = Field(description="Preguntas frecuentes relacionadas con el tema")
    intent_classification: list[KeywordIntent] = Field(
        description="Clasificación informacional/transaccional de cada keyword"
    )


## 4. Modelos: articles (`app/models/articles.py`)

`ArticleSection` con `heading_level` restringido a `H2`/`H3` (`Literal`). La IA genera un **borrador** (`ArticleDraft`, sin densidad); el servicio calcula la `keyword_density` real sobre el texto y construye la `ArticleResponse` final — así la densidad es un dato medido, no inventado por el modelo.


In [20]:
class ArticleRequest(BaseModel):
    """Entrada del endpoint de artículos."""
    main_keyword: str = Field(description="Keyword principal del artículo")
    secondary_keywords: list[str] = Field(default_factory=list,
                                          description="Keywords secundarias a integrar")
    word_count: int = Field(default=800, ge=300, le=2000,
                            description="Longitud objetivo del artículo en palabras")
    tone: str = Field(default="profesional", description="Tono de redacción")


class ArticleSection(BaseModel):
    """Sección del artículo con jerarquía SEO."""
    model_config = ConfigDict(extra="forbid")
    heading_level: Literal["H2", "H3"] = Field(description="Nivel del encabezado (H2 o H3)")
    title: str = Field(description="Texto del encabezado")
    content: str = Field(description="Contenido de la sección")


class ArticleDraft(BaseModel):
    """Lo que genera la IA (la densidad se calcula después en el servidor)."""
    model_config = ConfigDict(extra="forbid")
    title: str = Field(description="Título H1 del artículo, incluyendo la keyword principal")
    sections: list[ArticleSection] = Field(description="Secciones H2/H3 en orden")
    call_to_actions: list[str] = Field(description="Llamadas a la acción coherentes con el contenido")


class ArticleResponse(BaseModel):
    """Salida del endpoint de artículos."""
    model_config = ConfigDict(extra="forbid")
    title: str
    sections: list[ArticleSection]
    keyword_density: float = Field(description="Densidad de la keyword principal en % (calculada)")
    call_to_actions: list[str]


## 5. Modelos: metadata (`app/models/metadata.py`)

`MetaTitle` (máx. **60**) y `MetaDescription` (máx. **160**) con un `model_validator` que **trunca en límite de palabra y recalcula `char_count`**: el límite queda garantizado aunque la IA se exceda. La IA devuelve listas de textos (`MetadataAI`) y el servicio construye los objetos validados.


In [21]:
from pydantic import model_validator

META_TITLE_MAX = 60
META_DESCRIPTION_MAX = 160


def _truncar_en_palabra(texto: str, maximo: int) -> str:
    """Trunca al último espacio dentro del límite (o corte duro si no hay espacios)."""
    texto = texto.strip()
    if len(texto) <= maximo:
        return texto
    cortado = texto[:maximo]
    if " " in cortado:
        cortado = cortado.rsplit(" ", 1)[0]
    return cortado.rstrip(" ,;:.")


class MetadataRequest(BaseModel):
    """Entrada del endpoint de metadatos."""
    article_title: str = Field(description="Título del artículo")
    main_keyword: str = Field(description="Keyword principal a incluir en los metadatos")
    article_excerpt: str = Field(description="Extracto o resumen del artículo")


class MetaTitle(BaseModel):
    """Meta title con límite de 60 caracteres garantizado."""
    model_config = ConfigDict(extra="forbid")
    text: str
    char_count: int = 0

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.text = _truncar_en_palabra(self.text, META_TITLE_MAX)
        if not any(c.isalnum() for c in self.text):
            raise ValueError("Meta title vacío tras el truncado (la IA devolvió solo espacios/puntuación).")
        self.char_count = len(self.text)
        return self


class MetaDescription(BaseModel):
    """Meta description con límite de 160 caracteres garantizado."""
    model_config = ConfigDict(extra="forbid")
    text: str
    char_count: int = 0

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.text = _truncar_en_palabra(self.text, META_DESCRIPTION_MAX)
        if not any(c.isalnum() for c in self.text):
            raise ValueError("Meta description vacía tras el truncado (la IA devolvió solo espacios/puntuación).")
        self.char_count = len(self.text)
        return self


class MetadataAI(BaseModel):
    """Lo que genera la IA: 3-5 variantes de cada, como texto plano."""
    model_config = ConfigDict(extra="forbid")
    meta_titles: list[str] = Field(description="3 a 5 variantes de meta title (máx. 60 caracteres)")
    meta_descriptions: list[str] = Field(description="3 a 5 variantes de meta description (máx. 160 caracteres)")


class MetadataResponse(BaseModel):
    """Salida del endpoint de metadatos."""
    model_config = ConfigDict(extra="forbid")
    meta_titles: list[MetaTitle]
    meta_descriptions: list[MetaDescription]


## 6. Modelos: faqs (`app/models/faqs.py`)

`FAQ` anidada dentro de `FAQResponse`. El **JSON-LD FAQPage** se construye **en el servidor** con `json.dumps` a partir de las FAQs ya validadas (consejo de la plantilla): así el código para rich snippets siempre es JSON válido.


In [22]:
class FAQRequest(BaseModel):
    """Entrada del endpoint de FAQs."""
    article_content: str = Field(description="Contenido del artículo del que extraer FAQs")
    max_questions: int = Field(default=5, ge=1, le=10,
                               description="Número máximo de preguntas a extraer")


class FAQ(BaseModel):
    """Una pregunta frecuente con su respuesta (50-150 palabras)."""
    model_config = ConfigDict(extra="forbid")
    question: str = Field(description="Pregunta natural y relevante basada en el artículo")
    answer: str = Field(description="Respuesta concisa y útil, de 50 a 150 palabras")


class FAQList(BaseModel):
    """Lo que genera la IA (el JSON-LD se construye después en el servidor)."""
    model_config = ConfigDict(extra="forbid")
    faqs: list[FAQ]


class FAQResponse(BaseModel):
    """Salida del endpoint de FAQs."""
    model_config = ConfigDict(extra="forbid")
    faqs: list[FAQ]
    json_ld_schema: str = Field(description="Esquema JSON-LD FAQPage listo para insertar (cadena)")
    json_ld: dict = Field(description="El mismo esquema como objeto JSON, para consumo programático")


## 7. Modelos: social (`app/models/social.py`)

Un modelo por plataforma. `TwitterContent` garantiza el límite de **280 caracteres** con el mismo patrón de truncado. En `SocialResponse`, cada plataforma es opcional (`None` si no se solicitó).


In [23]:
TWITTER_MAX = 280
# Límites (aproximados) del resto de plataformas, para acotar el formato:
LINKEDIN_MAX = 3000
INSTAGRAM_MAX = 2200
FACEBOOK_MAX = 5000
# Máximo de hashtags por plataforma (pautas de los prompts, garantizadas por validador):
MAX_HASHTAGS = {"twitter": 3, "linkedin": 5, "instagram": 10, "facebook": 3}

Plataforma = Literal["twitter", "linkedin", "instagram", "facebook"]


class SocialRequest(BaseModel):
    """Entrada del endpoint de resúmenes sociales."""
    article_title: str = Field(description="Título del artículo o tema")
    article_content: str = Field(description="Contenido del artículo a resumir")
    target_platforms: list[Plataforma] = Field(
        default=["twitter", "linkedin", "instagram", "facebook"],
        description="Plataformas para las que generar contenido",
    )


class TwitterContent(BaseModel):
    """Post para Twitter/X: máximo 280 caracteres garantizado."""
    model_config = ConfigDict(extra="forbid")
    text: str = Field(description="Texto del tuit (incluye hashtags dentro del límite)")
    hashtags: list[str] = Field(description="Hashtags usados")
    char_count: int = 0

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.text = _truncar_en_palabra(self.text, TWITTER_MAX)
        self.char_count = len(self.text)
        # Cinturón y tirantes: el truncado garantiza el límite; esto lo hace explícito
        assert self.char_count <= TWITTER_MAX, "El tuit supera los 280 caracteres"
        self.hashtags = self.hashtags[: MAX_HASHTAGS["twitter"]]
        return self


class LinkedInContent(BaseModel):
    """Post para LinkedIn: profesional, con espacio para desarrollo (máx. ~3000)."""
    model_config = ConfigDict(extra="forbid")
    text: str = Field(description="Texto del post profesional")
    hashtags: list[str] = Field(description="Hashtags profesionales")

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.text = _truncar_en_palabra(self.text, LINKEDIN_MAX)
        self.hashtags = self.hashtags[: MAX_HASHTAGS["linkedin"]]
        return self


class InstagramContent(BaseModel):
    """Caption para Instagram: cercano, con emojis y hashtags (máx. 2200)."""
    model_config = ConfigDict(extra="forbid")
    caption: str = Field(description="Caption cercano con emojis")
    hashtags: list[str] = Field(description="Hashtags de descubrimiento")

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.caption = _truncar_en_palabra(self.caption, INSTAGRAM_MAX)
        self.hashtags = self.hashtags[: MAX_HASHTAGS["instagram"]]
        return self


class FacebookContent(BaseModel):
    """Post para Facebook: conversacional, invita a comentar (máx. ~5000)."""
    model_config = ConfigDict(extra="forbid")
    text: str = Field(description="Texto conversacional")
    hashtags: list[str] = Field(description="Hashtags (pocos)")

    @model_validator(mode="after")
    def _aplicar_limite(self):
        self.text = _truncar_en_palabra(self.text, FACEBOOK_MAX)
        self.hashtags = self.hashtags[: MAX_HASHTAGS["facebook"]]
        return self


class SocialResponse(BaseModel):
    """Salida del endpoint social: solo las plataformas solicitadas."""
    model_config = ConfigDict(extra="forbid")
    twitter: TwitterContent | None = None
    linkedin: LinkedInContent | None = None
    instagram: InstagramContent | None = None
    facebook: FacebookContent | None = None


## 8. Servicio SEO (`app/services/seo_service.py`)

El corazón de la API: una función genérica `_generate()` con `client.chat.completions.parse(...)` (**salidas estructuradas** ya validadas contra el modelo Pydantic) y una función por endpoint con su **prompt específico**. La densidad de keywords y el JSON-LD se calculan aquí, en el servidor.


In [24]:
import json
import logging
import re
from typing import TypeVar

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
srv_logger = logging.getLogger("seo_service")

T = TypeVar("T", bound=BaseModel)


def _generate(system_prompt: str, user_prompt: str, response_model: type[T]) -> T:
    """Llamada genérica con salida estructurada (chat.completions.parse)."""
    client = get_client()
    completion = client.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=response_model,
    )
    parsed = completion.choices[0].message.parsed
    if parsed is None:
        raise ValueError("El modelo no devolvió una respuesta estructurada válida.")
    return parsed


# ---------------------------------------------------------------- keywords ---
def generate_keywords(request: KeywordRequest) -> KeywordResponse:
    """Genera keywords semilla, long-tail, preguntas y clasificación de intención."""
    system = (
        "Eres un especialista SEO senior. Generas keywords útiles y realistas para "
        "posicionamiento en buscadores, siempre en el idioma solicitado."
    )
    user = (
        f"Tema principal: {request.topic}\n"
        f"Industria: {request.industry}\n"
        f"Idioma: {request.language}\n\n"
        "Genera:\n"
        "- 5 a 8 seed_keywords: términos base de 1-2 palabras.\n"
        "- 6 a 10 long_tail_keywords: variantes específicas de 3+ palabras con "
        "intención clara.\n"
        "- 5 a 8 questions: preguntas reales que la gente busca sobre el tema, "
        "redactadas también en el idioma indicado (TODO el contenido —keywords y "
        "preguntas— debe estar en ese idioma, sin mezclar).\n"
        "- intent_classification: clasifica CADA seed y long-tail keyword como "
        "'informacional' (busca aprender) o 'transaccional' (busca comprar/contratar). "
        "Sé preciso: 'qué es X' es informacional; 'comprar X barato' es transaccional.\n"
        "IMPORTANTE: intent_classification debe cubrir TODAS las keywords de "
        "seed_keywords y long_tail_keywords, sin omitir ninguna, usando EXACTAMENTE el "
        "mismo texto de cada keyword (misma grafía) para que las listas queden alineadas. "
        "Cuando tenga sentido como término de búsqueda, incluye también las preguntas de "
        "questions en intent_classification (las preguntas suelen ser informacionales)."
    )
    return _generate(system, user, KeywordResponse)


# ---------------------------------------------------------------- articles ---
def _calcular_densidad(texto: str, keyword: str) -> float:
    """% de apariciones de la keyword (como palabra/frase COMPLETA) sobre el total de palabras.

    Usa límites de palabra en la regex para no contar coincidencias parciales dentro de
    otras palabras (p. ej. 'café' no debe contar dentro de 'cafetería').
    """
    palabras = re.findall(r"\w+", texto.lower(), flags=re.UNICODE)
    if not palabras:
        return 0.0
    patron = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"
    apariciones = len(re.findall(patron, texto.lower(), flags=re.UNICODE))
    # Fórmula estándar: (apariciones de la frase / total de palabras) * 100
    return round(100 * apariciones / len(palabras), 2)


def _validar_jerarquia(sections: list["ArticleSection"]) -> None:
    """Comprueba la jerarquía SEO: sin H3 antes del primer H2 (todo H3 cuelga de un H2 previo)."""
    if not sections:
        raise ValueError("El artículo no tiene secciones.")
    if sections[0].heading_level != "H2":
        raise ValueError(
            "Estructura SEO inválida: la primera sección debe ser un H2 "
            f"(la IA devolvió un {sections[0].heading_level})."
        )
    # Escaneo explícito: ningún H3 puede aparecer sin un H2 previo en el recorrido
    visto_h2 = False
    for i, seccion in enumerate(sections):
        if seccion.heading_level == "H2":
            visto_h2 = True
        elif seccion.heading_level == "H3" and not visto_h2:
            raise ValueError(f"Estructura SEO inválida: la sección {i+1} es un H3 sin H2 previo.")
    n_h2 = sum(1 for s in sections if s.heading_level == "H2")
    if n_h2 < 3:
        srv_logger.warning("El artículo tiene solo %d secciones H2 (se pedían >=3).", n_h2)


def generate_article(request: ArticleRequest) -> ArticleResponse:
    """Genera un artículo SEO con H1/H2/H3, densidad natural y CTAs."""
    system = (
        "Eres un redactor SEO experto. Escribes artículos bien estructurados, útiles y "
        "naturales, en español. PROHIBIDO el keyword stuffing: integra las keywords con "
        "naturalidad (densidad objetivo 1-2%, nunca más de 3%)."
    )
    secundarias = ", ".join(request.secondary_keywords) or "(ninguna)"
    user = (
        f"Keyword principal: {request.main_keyword}\n"
        f"Keywords secundarias: {secundarias}\n"
        f"Longitud objetivo: ~{request.word_count} palabras\n"
        f"Tono: {request.tone}\n\n"
        "Estructura requerida:\n"
        "- title: el H1, atractivo y conteniendo la keyword principal.\n"
        "- sections: entre 3 y 6 secciones de nivel H2 como MÍNIMO 3. Usa H3 SOLO como "
        "subapartado del H2 inmediatamente anterior (nunca un H3 antes del primer H2 ni "
        "colgando de otro H3). El orden de la lista es el orden del artículo.\n"
        "- La primera sección debe cubrir la intención de búsqueda principal.\n"
        "- call_to_actions: 2-3 CTAs coherentes con el contenido (no genéricos)."
    )
    draft = _generate(system, user, ArticleDraft)
    _validar_jerarquia(draft.sections)
    texto_completo = draft.title + " " + " ".join(s.title + " " + s.content for s in draft.sections)
    densidad = _calcular_densidad(texto_completo, request.main_keyword)
    return ArticleResponse(
        title=draft.title,
        sections=draft.sections,
        keyword_density=densidad,
        call_to_actions=draft.call_to_actions,
    )


# ---------------------------------------------------------------- metadata ---
def generate_metadata(request: MetadataRequest) -> MetadataResponse:
    """Genera 3-5 meta titles (<=60) y 3-5 meta descriptions (<=160)."""
    system = (
        "Eres un especialista en SEO on-page y copywriting. Escribes metadatos "
        "persuasivos que maximizan el CTR en los resultados de búsqueda, en español."
    )
    user = (
        f"Título del artículo: {request.article_title}\n"
        f"Keyword principal: {request.main_keyword}\n"
        f"Extracto: {request.article_excerpt}\n\n"
        "Genera:\n"
        f"- meta_titles: 4 variantes de MÁXIMO {META_TITLE_MAX} caracteres, incluyendo "
        "la keyword principal, con lenguaje persuasivo (números, beneficios, urgencia "
        "moderada).\n"
        f"- meta_descriptions: 4 variantes de MÁXIMO {META_DESCRIPTION_MAX} caracteres, "
        "incluyendo la keyword principal y una llamada a la acción.\n"
        "TODAS las variantes (titles Y descriptions) deben contener la keyword principal "
        "TAL CUAL (misma grafía, sin variar).\n"
        "Cuenta los caracteres con cuidado: superar el límite corta el texto en Google."
    )
    ai = _generate(system, user, MetadataAI)
    # Requisito 3-5 variantes: si la IA devuelve menos de 3, reintentamos una vez
    if len(ai.meta_titles) < 3 or len(ai.meta_descriptions) < 3:
        srv_logger.warning(
            "La IA devolvió %d titles y %d descriptions (<3); reintentando una vez.",
            len(ai.meta_titles), len(ai.meta_descriptions),
        )
        ai = _generate(system, user + "\nRecuerda: EXACTAMENTE 4 variantes de cada.", MetadataAI)
        if len(ai.meta_titles) < 3 or len(ai.meta_descriptions) < 3:
            srv_logger.warning("Tras el reintento siguen faltando variantes; se devuelven las disponibles.")
    # Verificación posterior: cada variante debe contener la keyword principal
    kw = request.main_keyword.lower()
    sin_kw_titles = [t for t in ai.meta_titles[:5] if kw not in t.lower()]
    sin_kw_descs = [d for d in ai.meta_descriptions[:5] if kw not in d.lower()]
    if sin_kw_titles or sin_kw_descs:
        srv_logger.warning(
            "Variantes sin la keyword principal '%s': %d titles, %d descriptions.",
            request.main_keyword, len(sin_kw_titles), len(sin_kw_descs),
        )
    # Los validadores de MetaTitle/MetaDescription garantizan el límite aunque la IA se pase
    return MetadataResponse(
        meta_titles=[MetaTitle(text=t) for t in ai.meta_titles[:5]],
        meta_descriptions=[MetaDescription(text=d) for d in ai.meta_descriptions[:5]],
    )


# -------------------------------------------------------------------- faqs ---
def _esquema_json_ld(faqs: list[FAQ]) -> dict:
    """Esquema JSON-LD FAQPage como objeto (dict), a partir de FAQs ya validadas."""
    return {
        "@context": "https://schema.org",
        "@type": "FAQPage",
        "mainEntity": [
            {
                "@type": "Question",
                "name": f.question,
                "acceptedAnswer": {"@type": "Answer", "text": f.answer},
            }
            for f in faqs
        ],
    }


def _construir_json_ld(faqs: list[FAQ]) -> str:
    """El mismo esquema serializado como cadena JSON-LD lista para insertar."""
    esquema = _esquema_json_ld(faqs)
    try:
        resultado = json.dumps(esquema, ensure_ascii=False, indent=2)
        json.loads(resultado)   # autocomprobación: debe ser JSON parseable
        return resultado
    except (TypeError, ValueError) as e:
        srv_logger.error("Error construyendo el JSON-LD: %s", e)
        raise ValueError(f"No se pudo construir el esquema JSON-LD: {e}")


def extract_faqs(request: FAQRequest) -> FAQResponse:
    """Extrae FAQs del artículo y genera el JSON-LD FAQPage en el servidor."""
    system = (
        "Eres un especialista SEO en rich snippets. Extraes de un artículo las preguntas "
        "que la gente haría realmente en Google, con respuestas útiles y autocontenidas, "
        "en español."
    )
    user = (
        f"Artículo:\n{request.article_content}\n\n"
        f"Extrae como máximo {request.max_questions} preguntas frecuentes RELEVANTES y "
        "naturales basadas en este artículo. Para cada una escribe una respuesta concisa "
        "y útil de ENTRE 50 Y 150 PALABRAS (cuenta las palabras: ni menos de 50 ni más "
        "de 150), comprensible sin leer el artículo."
    )
    ai = _generate(system, user, FAQList)

    def _faqs_validas(lista: list[FAQ]) -> list[FAQ]:
        """Descarta FAQs con pregunta o respuesta vacías (sin caracteres alfanuméricos)."""
        validas = []
        for f in lista:
            if any(c.isalnum() for c in f.question) and any(c.isalnum() for c in f.answer):
                validas.append(f)
            else:
                srv_logger.warning("FAQ descartada por vacía: %r", f.question[:50])
        return validas

    faqs = _faqs_validas(ai.faqs)[: request.max_questions]

    # Validación posterior del rango 50-150 palabras; un reintento si alguna se sale
    fuera_de_rango = [f.question for f in faqs if not (50 <= len(f.answer.split()) <= 150)]
    if fuera_de_rango:
        srv_logger.warning("FAQs fuera del rango 50-150 palabras: %s. Reintentando una vez.",
                           fuera_de_rango)
        refuerzo = ("\nATENCIÓN: en el intento anterior estas preguntas tuvieron respuestas "
                    f"fuera del rango 50-150 palabras: {fuera_de_rango}. Corrígelo: TODAS "
                    "las respuestas deben tener entre 50 y 150 palabras exactamente.")
        ai = _generate(system, user + refuerzo, FAQList)
        faqs = _faqs_validas(ai.faqs)[: request.max_questions]
        aun_fuera = [f.question for f in faqs if not (50 <= len(f.answer.split()) <= 150)]
        if aun_fuera:
            srv_logger.warning("Tras el reintento siguen fuera de rango: %s. Se devuelven igualmente.",
                               aun_fuera)

    esquema = _esquema_json_ld(faqs)
    return FAQResponse(faqs=faqs, json_ld_schema=_construir_json_ld(faqs), json_ld=esquema)


# ------------------------------------------------------------------ social ---
_PROMPTS_PLATAFORMA = {
    "twitter": (
        TwitterContent,
        f"Twitter/X: un tuit de MÁXIMO {TWITTER_MAX} caracteres CONTANDO los hashtags "
        "(déjalos dentro del texto). Directo, con gancho, 1-3 hashtags, y una llamada a "
        "la acción breve (p. ej. lee el artículo).",
    ),
    "linkedin": (
        LinkedInContent,
        "LinkedIn: post profesional de 3-5 párrafos cortos con salto de línea, gancho "
        "inicial, aporte de valor, 3-5 hashtags profesionales al final y CTA de debate "
        "o lectura.",
    ),
    "instagram": (
        InstagramContent,
        "Instagram: caption cercano y visual, con emojis, frases cortas, 5-10 hashtags "
        "de descubrimiento y CTA típico de la plataforma (guarda este post, link en bio).",
    ),
    "facebook": (
        FacebookContent,
        "Facebook: texto conversacional de 2-3 párrafos que invite a comentar y "
        "compartir, tono cercano, 1-3 hashtags y una pregunta final a la audiencia.",
    ),
}


def generate_social(request: SocialRequest) -> SocialResponse:
    """Genera contenido adaptado a cada plataforma solicitada (una llamada por red)."""
    system = (
        "Eres un social media manager experto. Adaptas un mismo contenido al tono, "
        "longitud y formato de cada red social, en español."
    )
    base = (
        f"Título del artículo: {request.article_title}\n"
        f"Contenido:\n{request.article_content[:4000]}\n\n"
        "Genera el contenido para la siguiente plataforma.\n"
    )
    resultados = {}
    for plataforma in dict.fromkeys(request.target_platforms):   # únicas, en orden
        modelo, instrucciones = _PROMPTS_PLATAFORMA[plataforma]
        srv_logger.info("Generando contenido para %s", plataforma)
        try:
            resultados[plataforma] = _generate(system, base + instrucciones, modelo)
        except Exception as e:
            # Fallo aislado de una red: esa plataforma queda a None y el resto continúa
            srv_logger.error("Fallo generando contenido para %s: %s: %s",
                             plataforma, type(e).__name__, e)
            resultados[plataforma] = None
    if all(v is None for v in resultados.values()):
        raise ValueError("No se pudo generar contenido para ninguna plataforma solicitada.")
    return SocialResponse(**resultados)


## 9. Routers (`app/routers/*.py`)

Un `APIRouter` por funcionalidad, con el patrón de la plantilla: recibir el Request, llamar al servicio, devolver el Response, y **try/except** que convierte errores de OpenAI en HTTP 502 y errores de validación en HTTP 500 con detalle.


In [25]:
from fastapi import APIRouter, HTTPException
from openai import OpenAIError

# ----------------------------------------------- app/routers/keywords.py ---
router_keywords = APIRouter(prefix="/api/keywords", tags=["keywords"])


@router_keywords.post("/generate", response_model=KeywordResponse)
def endpoint_keywords(request: KeywordRequest) -> KeywordResponse:
    try:
        return generate_keywords(request)
    except OpenAIError as error:
        raise HTTPException(status_code=502, detail=f"Error de la API de OpenAI: {error}") from error
    except ValueError as error:
        raise HTTPException(status_code=500, detail=str(error)) from error


# ----------------------------------------------- app/routers/articles.py ---
router_articles = APIRouter(prefix="/api/articles", tags=["articles"])


@router_articles.post("/generate", response_model=ArticleResponse)
def endpoint_articles(request: ArticleRequest) -> ArticleResponse:
    try:
        return generate_article(request)
    except OpenAIError as error:
        raise HTTPException(status_code=502, detail=f"Error de la API de OpenAI: {error}") from error
    except ValueError as error:
        raise HTTPException(status_code=500, detail=str(error)) from error


# ----------------------------------------------- app/routers/metadata.py ---
router_metadata = APIRouter(prefix="/api/metadata", tags=["metadata"])


@router_metadata.post("/generate", response_model=MetadataResponse)
def endpoint_metadata(request: MetadataRequest) -> MetadataResponse:
    try:
        return generate_metadata(request)
    except OpenAIError as error:
        raise HTTPException(status_code=502, detail=f"Error de la API de OpenAI: {error}") from error
    except ValueError as error:
        raise HTTPException(status_code=500, detail=str(error)) from error


# --------------------------------------------------- app/routers/faqs.py ---
router_faqs = APIRouter(prefix="/api/faqs", tags=["faqs"])


@router_faqs.post("/extract", response_model=FAQResponse)
def endpoint_faqs(request: FAQRequest) -> FAQResponse:
    try:
        return extract_faqs(request)
    except OpenAIError as error:
        raise HTTPException(status_code=502, detail=f"Error de la API de OpenAI: {error}") from error
    except ValueError as error:
        raise HTTPException(status_code=500, detail=str(error)) from error


# ------------------------------------------------- app/routers/social.py ---
router_social = APIRouter(prefix="/api/social", tags=["social"])


@router_social.post(
    "/summaries",
    response_model=SocialResponse,
    description=(
        "Genera contenido adaptado a cada plataforma solicitada. Las plataformas no "
        "solicitadas llegan a null. IMPORTANTE: si la generación de una plataforma "
        "concreta falla, esa plataforma también llega a null (fallo parcial) y el resto "
        "se devuelve con normalidad; solo si fallan todas se responde con error."
    ),
)
def endpoint_social(request: SocialRequest) -> SocialResponse:
    try:
        return generate_social(request)
    except OpenAIError as error:
        raise HTTPException(status_code=502, detail=f"Error de la API de OpenAI: {error}") from error
    except ValueError as error:
        raise HTTPException(status_code=500, detail=str(error)) from error

print("✅ Routers creados: keywords, articles, metadata, faqs, social")


✅ Routers creados: keywords, articles, metadata, faqs, social


## 10. App FastAPI (`app/main.py`)

Monta los 5 routers y un endpoint raíz de salud. La **documentación OpenAPI/Swagger** queda disponible automáticamente en `/docs`.


In [26]:
from fastapi import FastAPI

app = FastAPI(
    title="SEO Content API",
    description=(
        "API REST de generación de contenido SEO con IA: keywords, artículos, "
        "metadatos, FAQs con JSON-LD y resúmenes para redes sociales."
    ),
    version="1.0.0",
)

app.include_router(router_keywords)
app.include_router(router_articles)
app.include_router(router_metadata)
app.include_router(router_faqs)
app.include_router(router_social)


@app.get("/", tags=["health"])
def root() -> dict:
    return {"status": "ok", "docs": "/docs"}


print(f"✅ App FastAPI creada con {len(app.routes)} rutas.")


✅ App FastAPI creada con 10 rutas.


## 11. Arrancar el servidor en localhost

Levantamos **uvicorn** en un hilo en segundo plano (en un proyecto real: `uvicorn app.main:app --reload`). Comprobamos con un socket que el puerto acepta conexiones antes de continuar. Con el servidor en marcha, abre **`http://127.0.0.1:8010/docs`** en el navegador para ver Swagger.


In [27]:
import socket
import threading
import time
import uvicorn

_servidor_api = None
_error_servidor = None


def _arrancar_api():
    global _servidor_api, _error_servidor
    try:
        config = uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="warning")
        _servidor_api = uvicorn.Server(config)
        _servidor_api.install_signal_handlers = lambda: None
        _servidor_api.run()
    except Exception as e:
        _error_servidor = e


def _puerto_abierto(host: str, puerto: int, timeout: float = 0.5) -> bool:
    try:
        with socket.create_connection((host, puerto), timeout=timeout):
            return True
    except OSError:
        return False


if not _puerto_abierto(API_HOST, API_PORT):
    threading.Thread(target=_arrancar_api, daemon=True).start()
    for _ in range(20):                      # hasta ~5 s de margen de arranque
        if _puerto_abierto(API_HOST, API_PORT):
            break
        time.sleep(0.25)

if _error_servidor:
    print(f"❌ El servidor no pudo arrancar: {type(_error_servidor).__name__}: {_error_servidor}")
elif _puerto_abierto(API_HOST, API_PORT):
    print(f"✅ API escuchando en {API_URL}")
    print(f"   Swagger: {API_URL}/docs")
else:
    print("❌ El puerto no responde tras el arranque. Revisa si ya hay algo usando el puerto.")


✅ API escuchando en http://127.0.0.1:8010
   Swagger: http://127.0.0.1:8010/docs


## 12. Demostración de los 5 endpoints

Flujo completo encadenado con datos reales: **keywords → artículo → metadatos → FAQs → social**. Cada demo llama a la API por HTTP (como lo haría cualquier cliente). Requiere el servidor arrancado y tu clave en el `.env`.


In [28]:
import requests

TEMA = "café de especialidad"

# ---- 1) POST /api/keywords/generate ----
r = requests.post(f"{API_URL}/api/keywords/generate", json={
    "topic": TEMA, "industry": "hostelería", "language": "es",
}, timeout=120)
r.raise_for_status()
kw = r.json()
print("── /api/keywords/generate ──")
print("Seed:", kw["seed_keywords"])
print("Long-tail:", kw["long_tail_keywords"][:4], "...")
print("Preguntas:", kw["questions"][:3], "...")
print("Intención (muestra):", kw["intent_classification"][:4])


2026-07-25 11:54:02,157 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


── /api/keywords/generate ──
Seed: ['café de especialidad', 'cafetería', 'barista', 'tostador', 'cafetera espresso', 'molino profesional', 'tueste medio']
Long-tail: ['comprar café de especialidad online', 'proveedores de café de especialidad para cafeterías', 'qué es el café de especialidad', 'cursos de barista para hostelería'] ...
Preguntas: ['¿Qué diferencia hay entre café de especialidad y comercial?', '¿Cómo elegir un buen proveedor de café para mi cafetería?', '¿Cómo calibrar el molino de café para espresso?'] ...
Intención (muestra): [{'keyword': 'café de especialidad', 'intent': 'informacional'}, {'keyword': 'cafetería', 'intent': 'transaccional'}, {'keyword': 'barista', 'intent': 'informacional'}, {'keyword': 'tostador', 'intent': 'transaccional'}]


In [29]:
# ---- 2) POST /api/articles/generate (usa las keywords del paso anterior) ----
r = requests.post(f"{API_URL}/api/articles/generate", json={
    "main_keyword": kw["seed_keywords"][0],
    "secondary_keywords": kw["long_tail_keywords"][:3],
    "word_count": 700,
    "tone": "profesional cercano",
}, timeout=300)
r.raise_for_status()
art = r.json()
print("── /api/articles/generate ──")
print("H1:", art["title"])
for s in art["sections"]:
    print(f"  [{s['heading_level']}] {s['title']}")
print("Densidad keyword:", art["keyword_density"], "%")
print("CTAs:", art["call_to_actions"])


2026-07-25 11:54:47,822 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


── /api/articles/generate ──
H1: Café de especialidad: qué es, cómo elegirlo y dónde comprarlo online
  [H2] Qué es el café de especialidad y por qué importa
  [H2] Cómo elegir tu café de especialidad: origen, proceso y tueste
  [H3] Origen y terroir
  [H3] Procesamiento del grano
  [H3] Tueste y frescura
  [H3] Molienda y método
  [H2] Comprar café de especialidad online: criterios para acertar
  [H2] Proveedores de café de especialidad para cafeterías: qué evaluar
  [H2] Errores comunes y consejos de preparación
Densidad keyword: 1.43 %
CTAs: ['¿Listo para dar el salto? Elige un origen, verifica la fecha de tueste y realiza tu primer pedido para comprar café de especialidad online en un tostador de tu país.', 'Si gestionas una barra, solicita muestras a dos proveedores de café de especialidad para cafeterías y programa una cata a ciegas con tu equipo esta misma semana.', '¿Prefieres descubrir nuevos perfiles sin decidir cada mes? Apúntate a una suscripción de café de especialidad con

In [30]:
# ---- 3) POST /api/metadata/generate ----
extracto = art["sections"][0]["content"][:400]
r = requests.post(f"{API_URL}/api/metadata/generate", json={
    "article_title": art["title"],
    "main_keyword": kw["seed_keywords"][0],
    "article_excerpt": extracto,
}, timeout=120)
r.raise_for_status()
meta = r.json()
print("── /api/metadata/generate ──")
for t in meta["meta_titles"]:
    print(f"  [title {t['char_count']:>2} ch] {t['text']}")
for d in meta["meta_descriptions"]:
    print(f"  [desc  {d['char_count']:>3} ch] {d['text']}")


2026-07-25 11:55:13,625 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


── /api/metadata/generate ──
  [title 52 ch] café de especialidad: 5 claves para elegir y comprar
  [title 49 ch] café de especialidad: guía rápida + dónde comprar
  [title 51 ch] café de especialidad: sabor top y compra online hoy
  [title 53 ch] café de especialidad: calidad 80+ SCA y dónde comprar
  [desc  140 ch] Descubre qué es el café de especialidad, cómo elegir granos 80+ SCA y dónde comprar online. Mejora tu taza desde hoy. Entra y elige el tuyo.
  [desc  140 ch] Aprende a identificar café de especialidad: puntuación 80+, trazabilidad y tueste óptimo. Compra online con confianza. Lee la guía y decide.
  [desc  140 ch] ¿Buscas café de especialidad? Compara orígenes, perfiles y precios. Aprende a elegir bien y compra online hoy. Entra ahora y mejora tu taza.
  [desc  133 ch] Todo sobre café de especialidad: qué es, estándares SCA 80+, diferencias con café comercial y dónde comprar. Lee la guía y elige hoy.


In [31]:
# ---- 4) POST /api/faqs/extract (sobre el artículo generado) ----
texto_articulo = art["title"] + "\n\n" + "\n\n".join(
    f"{s['title']}\n{s['content']}" for s in art["sections"]
)
r = requests.post(f"{API_URL}/api/faqs/extract", json={
    "article_content": texto_articulo, "max_questions": 4,
}, timeout=180)
r.raise_for_status()
faqs = r.json()
print("── /api/faqs/extract ──")
for f in faqs["faqs"]:
    palabras = len(f["answer"].split())
    print(f"  ❓ {f['question']}  ({palabras} palabras de respuesta)")
print("\nJSON-LD (inicio):")
print(faqs["json_ld_schema"][:300], "...")


2026-07-25 11:55:35,027 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


── /api/faqs/extract ──
  ❓ ¿Qué es el café de especialidad y en qué se diferencia del comercial?  (101 palabras de respuesta)
  ❓ ¿Cómo elijo un café de especialidad según mi gusto y método de preparación?  (101 palabras de respuesta)
  ❓ ¿Qué debo revisar al comprar café de especialidad online?  (104 palabras de respuesta)
  ❓ ¿Cuáles son los errores más comunes al preparar café de especialidad y cómo evitarlos?  (98 palabras de respuesta)

JSON-LD (inicio):
{
  "@context": "https://schema.org",
  "@type": "FAQPage",
  "mainEntity": [
    {
      "@type": "Question",
      "name": "¿Qué es el café de especialidad y en qué se diferencia del comercial?",
      "acceptedAnswer": {
        "@type": "Answer",
        "text": "El café de especialidad es aquel ...


In [32]:
# ---- 5) POST /api/social/summaries ----
r = requests.post(f"{API_URL}/api/social/summaries", json={
    "article_title": art["title"],
    "article_content": texto_articulo,
    "target_platforms": ["twitter", "linkedin", "instagram", "facebook"],
}, timeout=300)
r.raise_for_status()
soc = r.json()
print("── /api/social/summaries ──")
if soc["twitter"]:
    print(f"🐦 Twitter ({len(soc['twitter']['text'])}/280 ch): {soc['twitter']['text']}\n")
if soc["linkedin"]:
    print(f"💼 LinkedIn: {soc['linkedin']['text'][:180]}...\n   {soc['linkedin']['hashtags']}\n")
if soc["instagram"]:
    print(f"📸 Instagram: {soc['instagram']['caption'][:150]}...\n   {soc['instagram']['hashtags']}\n")
if soc["facebook"]:
    print(f"👥 Facebook: {soc['facebook']['text'][:180]}...")


2026-07-25 11:55:35,060 [INFO] seo_service: Generando contenido para twitter
2026-07-25 11:55:53,458 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-25 11:55:53,466 [INFO] seo_service: Generando contenido para linkedin
2026-07-25 11:56:13,019 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-25 11:56:13,023 [INFO] seo_service: Generando contenido para instagram
2026-07-25 11:56:35,853 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-25 11:56:35,857 [INFO] seo_service: Generando contenido para facebook
2026-07-25 11:56:51,449 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


── /api/social/summaries ──
🐦 Twitter (238/280 ch): ¿Café de especialidad? No es moda: +80 SCA, trazabilidad y tueste reciente para tazas limpias y complejas. Aprende a elegir por origen, proceso y tueste y dónde comprar online sin fallar. Lee el artículo. #CaféDeEspecialidad #CoffeeLovers

💼 LinkedIn: El café de especialidad no es solo “sabe rico”: es calidad medible, trazabilidad y tueste reciente. Hablamos de cafés evaluados >80 SCA, seleccionados a mano y tostados para resalt...
   ['#CaféDeEspecialidad', '#Barismo', '#Tostadores', '#Ecommerce', '#GestiónDeCafeterías']

📸 Instagram: ¿Café de especialidad? ☕️💎 Más que “rico”: +80 SCA, trazabilidad de finca a taza y tueste reciente para tazas limpias, dulces y complejas.

Cómo elegi...
   ['#cafedeespecialidad', '#coffee', '#coffeelovers', '#barista', '#cafe', '#espresso', '#v60', '#tostadores', '#catasdecafe', '#thirdwavecoffee']

👥 Facebook: ¿Qué es el café de especialidad? Mucho más que “rico”: son cafés evaluados por catadores co

## 13. Documentación automática (Swagger)

Con el servidor en marcha, la documentación **OpenAPI** interactiva está en:

- **Swagger UI:** `http://127.0.0.1:8010/docs` — prueba los 5 endpoints desde el navegador.
- **ReDoc:** `http://127.0.0.1:8010/redoc`
- **Esquema OpenAPI (JSON):** `http://127.0.0.1:8010/openapi.json`

La celda siguiente comprueba que el esquema OpenAPI expone los 5 endpoints.


In [33]:
r = requests.get(f"{API_URL}/openapi.json", timeout=10)
r.raise_for_status()
rutas = sorted(p for p in r.json()["paths"] if p.startswith("/api/"))
print("Endpoints documentados en OpenAPI/Swagger:")
for ruta in rutas:
    print("  •", ruta)
assert len(rutas) == 5, "Deberían ser exactamente 5 endpoints /api/*"
print("\n✅ Los 5 endpoints están documentados. Ábrelos en el navegador:", f"{API_URL}/docs")


Endpoints documentados en OpenAPI/Swagger:
  • /api/articles/generate
  • /api/faqs/extract
  • /api/keywords/generate
  • /api/metadata/generate
  • /api/social/summaries

✅ Los 5 endpoints están documentados. Ábrelos en el navegador: http://127.0.0.1:8010/docs
